In [ ]:
"""
서울의 기분 프로젝트 — 스트레스 해결책 크롤러 (카카오맵)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

SUSI 지수(Seoul Urban Stress Index) 4대 분류별 해결 장소 수집
- 환경 스트레스 → 공원, 숲, 자연 힐링 공간
- 교통 스트레스 → 역세권, 접근성 좋은 장소
- 소음 스트레스 → 조용한 카페, 도서관, 한적한 공간
- 안전 스트레스 → 밝고 사람 많은 안전한 장소

[사용법]
1. ★ API 키 입력
2. ★ 수집할 스트레스 태그 선택 (STRESS_TAGS)
3. 실행 → 자치구별 해결책 장소 CSV 저장
"""

import requests
import pandas as pd
import time
import os
import json
from datetime import datetime
from typing import List, Dict, Tuple

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# ★ ① 카카오 REST API 키 입력
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
KAKAO_API_KEY = "7d4d084bb53dd628a241624f63ecbf59"

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# ★ ② 수집할 스트레스 해결 태그 선택
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
STRESS_TAGS = {
    # 환경 스트레스 해결책
    "환경": {
        "keywords": [
            "공원", "한강공원", "산책로", "숲", "생태공원",
            "정원", "수목원", "식물원", "녹지", "힐링",
        ],
        "categories": ["AT4"],  # 관광명소
    },
    
    # 교통 스트레스 해결책
    "교통": {
        "keywords": [
            "역세권 카페", "역 앞", "지하철역", "환승역",
            "접근성 좋은", "역 5분", "주차 편한",
        ],
        "categories": ["CE7", "FD6"],  # 카페, 음식점
    },
    
    # 소음 스트레스 해결책
    "소음": {
        "keywords": [
            "조용한 카페", "도서관", "북카페", "한적한",
            "스터디카페", "독서실", "미술관", "박물관",
            "조용한", "차분한", "집중",
        ],
        "categories": ["CE7", "CT1"],  # 카페, 문화시설
    },
    
    # 안전 스트레스 해결책
    "안전": {
        "keywords": [
            "밝은 카페", "사람 많은", "번화가", "대형 쇼핑몰",
            "백화점", "복합문화공간", "안정", "개방", "느좋",  "안전쉼터", 
            "스마트안전존", "밤산책", "야경명소",
        ],
        "categories": [],
    },
}

# ★ 수집할 스트레스 분류 선택 (리스트로 여러 개 가능)
SELECTED_STRESS = ["안전"]  # 전체 수집
# SELECTED_STRESS = ["소음"]  # 특정 하나만 수집

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 설정
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
KEYWORD_SEARCH_URL  = "https://dapi.kakao.com/v2/local/search/keyword.json"
CATEGORY_SEARCH_URL = "https://dapi.kakao.com/v2/local/search/category.json"

OUTPUT_DIR = "stress_solutions"  # 결과 저장 폴더
PROGRESS_FILE = "kakao_stress_progress.json"  # 진행상황 저장

# 서울 25개 자치구 중심 좌표 (경도, 위도)
DISTRICT_COORDS = {
    "강남구": (127.0495, 37.5172), "강동구": (127.1238, 37.5301),
    "강북구": (127.0277, 37.6396), "강서구": (126.8496, 37.5510),
    "관악구": (126.9515, 37.4784), "광진구": (127.0823, 37.5385),
    "구로구": (126.8874, 37.4954), "금천구": (126.8956, 37.4568),
    "노원구": (127.0567, 37.6541), "도봉구": (127.0469, 37.6688),
    "동대문구": (127.0407, 37.5744), "동작구": (126.9396, 37.5121),
    "마포구": (126.9065, 37.5638), "서대문구": (126.9360, 37.5791),
    "서초구": (127.0325, 37.4836), "성동구": (127.0369, 37.5634),
    "성북구": (127.0200, 37.5894), "송파구": (127.1143, 37.5145),
    "양천구": (126.8687, 37.5170), "영등포구": (126.8963, 37.5264),
    "용산구": (126.9645, 37.5320), "은평구": (126.9291, 37.6026),
    "종로구": (126.9784, 37.5730), "중구":   (126.9996, 37.5636),
    "중랑구": (127.0927, 37.6063),
}

SEOUL_DISTRICTS = list(DISTRICT_COORDS.keys())


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Session 관리
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def make_session(api_key: str) -> requests.Session:
    s = requests.Session()
    s.headers.update({
        "Authorization": f"KakaoAK {api_key}",
        "Content-Type":  "application/json;charset=UTF-8",
    })
    return s


def test_api_key(session: requests.Session) -> bool:
    """API 키 유효성 검증"""
    print("🔑 카카오 API 키 검증 중...")
    if KAKAO_API_KEY in ("", "여기에_본인의_카카오_REST_API_키_입력"):
        print("  ❌ API 키가 입력되지 않았습니다.")
        print("     → 코드 상단에 본인의 카카오 REST API 키를 입력하세요.\n")
        return False
    
    try:
        resp = session.get(KEYWORD_SEARCH_URL, params={"query": "강남역", "size": 1}, timeout=10)
        if resp.status_code == 200:
            print("  ✅ API 키 정상\n")
            return True
        elif resp.status_code == 403:
            print("  ❌ 403 Forbidden")
            print("     → [카카오 로그인] 활성화 + [플랫폼] 설정 확인 필요")
            print("     → http://localhost 도메인 등록 필수\n")
        else:
            print(f"  ❌ HTTP {resp.status_code}\n")
    except Exception as e:
        print(f"  ❌ 오류: {e}\n")
    return False


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 진행상황 관리
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def load_progress() -> Dict:
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE, "r", encoding="utf-8") as f:
            return json.load(f)
    return {"completed": {}}


def save_progress(progress: Dict):
    with open(PROGRESS_FILE, "w", encoding="utf-8") as f:
        json.dump(progress, f, ensure_ascii=False, indent=2)


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 검색 함수
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def search_by_keyword(
    session: requests.Session,
    query: str,
    x: float = None,
    y: float = None,
    radius: int = 5000,
    page: int = 1,
) -> Tuple[List[Dict], bool]:
    """키워드 검색 (좌표 기반 반경 검색 가능)"""
    params = {
        "query": query,
        "page": page,
        "size": 15,
        "sort": "accuracy",
    }
    if x and y:
        params["x"] = x
        params["y"] = y
        params["radius"] = radius
    
    resp = session.get(KEYWORD_SEARCH_URL, params=params, timeout=10)
    if resp.status_code != 200:
        return [], False
    data = resp.json()
    return data.get("documents", []), not data["meta"]["is_end"]


def search_by_category(
    session: requests.Session,
    category_code: str,
    x: float,
    y: float,
    radius: int = 3000,
    page: int = 1,
) -> Tuple[List[Dict], bool]:
    """카테고리 코드 검색"""
    params = {
        "category_group_code": category_code,
        "x": x,
        "y": y,
        "radius": radius,
        "page": page,
        "size": 15,
        "sort": "distance",  # 거리순 (가까운 곳부터)
    }
    resp = session.get(CATEGORY_SEARCH_URL, params=params, timeout=10)
    if resp.status_code != 200:
        return [], False
    data = resp.json()
    return data.get("documents", []), not data["meta"]["is_end"]


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 데이터 파싱 및 품질 검증
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def parse_item(item: Dict, district: str, stress_type: str, tag: str) -> Dict:
    """검색 결과를 표준 형식으로 변환"""
    return {
        "자치구":      district,
        "스트레스분류": stress_type,
        "해결태그":    tag,
        "장소명":      item.get("place_name", ""),
        "카테고리":    item.get("category_name", ""),
        "도로명주소":  item.get("road_address_name", ""),
        "경도":       float(item.get("x", 0)),
        "위도":       float(item.get("y", 0)),
        "전화번호":    item.get("phone", ""),
        "플레이스URL": item.get("place_url", ""),
        "_카카오맵ID": item.get("id", ""),  # 중복 제거용 (출력 시 제외)
    }


def is_valid_place(place: Dict, district: str) -> bool:
    """장소 유효성 검증 (자치구 일치 여부 등)"""
    # 도로명주소 기준으로 검증 (없으면 통과)
    road_address = place.get("도로명주소", "")
    
    # 1. 도로명주소가 있으면 자치구 일치 여부 확인
    if road_address and district not in road_address:
        return False
    
    # 2. 필수 정보 존재 여부
    if not place.get("장소명") or not place.get("경도") or not place.get("위도"):
        return False
    
    # 3. 서울시 좌표 범위 검증 (대략적)
    lng, lat = place["경도"], place["위도"]
    if not (126.7 <= lng <= 127.2 and 37.4 <= lat <= 37.7):
        return False
    
    return True


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 자치구 × 스트레스 분류별 수집
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def crawl_district_stress(
    session: requests.Session,
    district: str,
    stress_type: str,
    config: Dict,
) -> List[Dict]:
    """한 자치구의 특정 스트레스 해결책 수집"""
    x, y = DISTRICT_COORDS[district]
    results = []
    seen_ids = set()
    
    print(f"\n  [{stress_type}] 수집 시작")
    
    # A. 키워드 검색
    for keyword in config["keywords"]:
        query = f"서울 {district} {keyword}"
        fetched = 0
        
        for page in range(1, 3):  # 최대 2페이지 (30건)
            try:
                items, has_next = search_by_keyword(session, query, x, y, page=page)
            except Exception as e:
                print(f"    [오류] {keyword}: {e}")
                break
            
            for item in items:
                pid = item.get("id", "")
                if pid in seen_ids:
                    continue
                
                place = parse_item(item, district, stress_type, f"키워드:{keyword}")
                
                # 유효성 검증
                if not is_valid_place(place, district):
                    continue
                
                seen_ids.add(pid)
                results.append(place)
                fetched += 1
            
            if not has_next:
                break
            time.sleep(0.3)
        
        print(f"    키워드[{keyword}]: {fetched}건")
    
    # B. 카테고리 검색
    for category_code in config["categories"]:
        fetched = 0
        
        for page in range(1, 3):
            try:
                items, has_next = search_by_category(session, category_code, x, y, page=page)
            except Exception as e:
                print(f"    [오류] 카테고리 {category_code}: {e}")
                break
            
            for item in items:
                pid = item.get("id", "")
                if pid in seen_ids:
                    continue
                
                place = parse_item(item, district, stress_type, f"카테고리:{category_code}")
                
                if not is_valid_place(place, district):
                    continue
                
                seen_ids.add(pid)
                results.append(place)
                fetched += 1
            
            if not has_next:
                break
            time.sleep(0.3)
        
        print(f"    카테고리[{category_code}]: {fetched}건")
    
    print(f"  [{stress_type}] 총 {len(results)}건 수집")
    return results


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 전체 수집 (자치구 × 스트레스 분류)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def crawl_all_solutions():
    """서울 25개 자치구 × 선택한 스트레스 분류별 해결책 수집"""
    session = make_session(KAKAO_API_KEY)
    
    if not test_api_key(session):
        print("❌ 크롤링을 중단합니다. API 키를 확인하세요.")
        return None
    
    # 출력 폴더 생성
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    # 진행상황 로드
    progress = load_progress()
    
    # 수집 시작
    all_results = {stress: [] for stress in SELECTED_STRESS}
    
    for i, district in enumerate(SEOUL_DISTRICTS, 1):
        print(f"\n{'='*60}")
        print(f"[{i:02d}/25] 🏘️  {district}")
        print(f"{'='*60}")
        
        for stress_type in SELECTED_STRESS:
            # 이미 수집 완료한 경우 건너뛰기
            key = f"{district}_{stress_type}"
            if progress["completed"].get(key):
                print(f"  [{stress_type}] ⏭️  이미 수집 완료 (건너뜀)")
                continue
            
            # 해당 스트레스 태그 설정 가져오기
            config = STRESS_TAGS[stress_type]
            
            # 수집
            results = crawl_district_stress(session, district, stress_type, config)
            all_results[stress_type].extend(results)
            
            # 진행상황 저장
            progress["completed"][key] = True
            save_progress(progress)
            
            time.sleep(0.5)
    
    # 스트레스 분류별 CSV 저장
    print(f"\n{'='*60}")
    print("📊 결과 저장 중...")
    print(f"{'='*60}")
    
    summary = []
    
    for stress_type in SELECTED_STRESS:
        if not all_results[stress_type]:
            print(f"  ⚠️  [{stress_type}] 수집된 데이터 없음")
            continue
        
        df = pd.DataFrame(all_results[stress_type])
        
        # 중복 제거 (카카오맵ID 기준)
        df.drop_duplicates(subset=["_카카오맵ID"], inplace=True)
        
        # 최종 출력 컬럼만 선택 (_카카오맵ID 제외)
        output_columns = [
            "자치구", "스트레스분류", "해결태그", "장소명", "카테고리",
            "도로명주소", "경도", "위도", "전화번호", "플레이스URL"
        ]
        df = df[output_columns]
        df.reset_index(drop=True, inplace=True)
        
        # 파일명: stress_solutions/환경_해결책_카카오.csv
        filename = f"{stress_type}_해결책_카카오.csv"
        filepath = os.path.join(OUTPUT_DIR, filename)
        df.to_csv(filepath, index=False, encoding="utf-8-sig")
        
        print(f"  ✅ [{stress_type}] {filepath} ({len(df)}건)")
        
        summary.append({
            "스트레스분류": stress_type,
            "수집건수": len(df),
            "자치구수": df["자치구"].nunique(),
            "파일경로": filepath,
        })
    
    # 요약 정보 저장
    summary_df = pd.DataFrame()
    if summary:
        summary_df = pd.DataFrame(summary)
        summary_path = os.path.join(OUTPUT_DIR, "_수집_요약.csv")
        summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")
        
        print(f"\n{'='*60}")
        print("✅ 전체 수집 완료!")
        print(f"{'='*60}")
        print(summary_df.to_string(index=False))
        print(f"\n📁 저장 위치: {os.path.abspath(OUTPUT_DIR)}/")
    
    return summary_df


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 실행
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
if __name__ == "__main__":
    print(f"""
{'='*60}
서울의 기분 프로젝트 — 스트레스 해결책 크롤러
{'='*60}
수집 대상 스트레스: {', '.join(SELECTED_STRESS)}
수집 범위: 서울 25개 자치구
{'='*60}
""")
    
    start_time = datetime.now()
    summary = crawl_all_solutions()
    end_time = datetime.now()
    
    if summary is not None:
        print(f"\n⏱️  소요 시간: {end_time - start_time}")



서울의 기분 프로젝트 — 스트레스 해결책 크롤러
수집 대상 스트레스: 안전
수집 범위: 서울 25개 자치구

🔑 카카오 API 키 검증 중...
  ✅ API 키 정상


[01/25] 🏘️  강남구

  [안전] 수집 시작
    키워드[밝은 카페]: 0건
    키워드[사람 많은]: 0건
    키워드[번화가]: 0건
    키워드[대형 쇼핑몰]: 4건
    키워드[백화점]: 25건
    키워드[복합문화공간]: 30건
    키워드[안정]: 1건
    키워드[개방]: 30건
    키워드[느좋]: 0건
    키워드[안전쉼터]: 0건
    키워드[스마트안전존]: 0건
    키워드[밤산책]: 0건
    키워드[야경명소]: 0건
  [안전] 총 90건 수집

[02/25] 🏘️  강동구

  [안전] 수집 시작
    키워드[밝은 카페]: 0건
    키워드[사람 많은]: 0건
    키워드[번화가]: 0건
    키워드[대형 쇼핑몰]: 0건
    키워드[백화점]: 22건
    키워드[복합문화공간]: 26건
    키워드[안정]: 2건
    키워드[개방]: 30건
    키워드[느좋]: 0건
    키워드[안전쉼터]: 0건
    키워드[스마트안전존]: 0건
    키워드[밤산책]: 0건
    키워드[야경명소]: 0건
  [안전] 총 80건 수집

[03/25] 🏘️  강북구

  [안전] 수집 시작
    키워드[밝은 카페]: 0건
    키워드[사람 많은]: 0건
    키워드[번화가]: 0건
    키워드[대형 쇼핑몰]: 0건
    키워드[백화점]: 19건
    키워드[복합문화공간]: 30건
    키워드[안정]: 1건
    키워드[개방]: 30건
    키워드[느좋]: 0건
    키워드[안전쉼터]: 0건
    키워드[스마트안전존]: 0건
    키워드[밤산책]: 0건
    키워드[야경명소]: 0건
  [안전] 총 80건 수집

[04/25] 🏘️  강서구

  [안전] 수집 시작
    키워드[밝은 카페]: 0건
    키

: 